# Predict on New Data with OctoPredictor

This notebook demonstrates how to use `OctoPredictor` to predict on new, unseen data after training an Octopus study.

**OctoPredictor vs OctoTestEvaluator:**

|                | OctoPredictor                                              | OctoTestEvaluator                              |
| -------------- | ---------------------------------------------------------- | ---------------------------------------------- |
| **Purpose**    | Predict on new, unseen data                                | Evaluate study results on held-out test folds  |
| **Data**       | You provide data explicitly                                | Uses stored train/test splits from the study   |
| **Prediction** | Ensemble average across all outer-split models             | Each model predicts only on its own test fold  |
| **Deployment** | Can be saved/loaded standalone (no study directory needed) | Requires the full study directory              |
| **Use case**   | Production inference, scoring new patients/samples         | Post-hoc model evaluation, internal validation |


## 1. Train a Study

We split the breast cancer dataset 80/20 — train the study on 80% and hold out 20% as "new" data for prediction.


In [ ]:
import os
import tempfile

import pandas as pd
from sklearn.model_selection import train_test_split

from octopus.example_data import load_breast_cancer_data
from octopus.modules import Octo
from octopus.study import OctoClassification
from octopus.types import ModelName

In [2]:
# Load the breast cancer dataset
df, features, targets = load_breast_cancer_data()

print(f"Full dataset: {df.shape[0]} samples, {len(features)} features")
print(f"Classes: {targets}")
print(f"Target distribution: {df['target'].value_counts().sort_index().to_dict()}")

Full dataset: 569 samples, 30 features
Classes: ['malignant', 'benign']
Target distribution: {0: 212, 1: 357}


In [3]:
# Split 80/20: train the study on train_df, predict on new_df later
train_df, new_df = train_test_split(df, test_size=0.2, random_state=42, stratify=df["target"])
train_df = train_df.reset_index(drop=True)
new_df = new_df.reset_index(drop=True)

print(f"Training set: {train_df.shape[0]} samples")
print(f"New data (held-out): {new_df.shape[0]} samples")

Training set: 455 samples
New data (held-out): 114 samples


In [5]:
# Create and fit a classification study (2 outer folds, 5 trials for speed)
study = OctoClassification(
    study_name="predict_example",
    studies_directory=os.environ.get("STUDIES_PATH", "./studies"),
    target_metric="AUCROC",
    feature_cols=features,
    target_col="target",
    sample_id_col="index",
    stratification_col="target",
    n_outer_splits=2,
    n_cpus=1,
    workflow=[
        Octo(
            description="step1_octo",
            task_id=0,
            depends_on=None,
            models=[ModelName.ExtraTreesClassifier],
            n_trials=5,
            n_inner_splits=3,
        ),
    ],
)

study.fit(data=train_df)
print(f"Study saved to: {study.output_path}")

INFO | CREATING_DATASPLITS | 455 rows, 455 groups (column: datasplit_group), StratifiedGroupKFold, 2 splits, seed 0
INFO | CREATING_DATASPLITS | Outer 0 created: Train/Dev - 228 rows, 228 groups | Test - 227 rows, 227 groups
INFO | CREATING_DATASPLITS | Outer 1 created: Train/Dev - 227 rows, 227 groups | Test - 228 rows, 228 groups
INFO | CREATING_DATASPLITS | Creating a local ray instance with 1 CPUs.


2026-04-10 16:47:25,585	INFO worker.py:2014 -- Started a local Ray instance. View the dashboard at http://127.0.0.1:8265 


INFO | CREATING_DATASPLITS | Preparing execution | 
Single outer split: False
Outer splits:      2
Available CPUs:    1
Workers:           1
CPUs/outer split:  1
Ray Nodes:
	Node __internal_head__, 127.0.0.1:  1.0 CPUs
INFO | PROCESSING | Running outer split: 0
INFO | PROCESSING | Processing workflow task: 0 | Input task: None | Module: octo | Description: step1_octo
INFO | PROCESSING | Running task 0 for outer split 0


/Users/M228393/github/Octopus_public/.venv/lib/python3.12/site-packages/ray/_private/worker.py:2062: FutureWarning: Tip: In future versions of Ray, Ray will no longer override accelerator visible devices env var if num_gpus=0 or num_gpus=None (default). To enable this behavior and turn off this error message, set RAY_ACCEL_ENV_VAR_OVERRIDE_ON_ZERO=0
  warnings.warn(


INFO | CREATING_DATASPLITS | 228 rows, 228 groups (column: datasplit_group), StratifiedGroupKFold, 3 splits, seed 0
INFO | CREATING_DATASPLITS | OUTER 0 SEQ TBD 0 created: Train - 152 rows, 152 groups | Dev - 76 rows, 76 groups
INFO | CREATING_DATASPLITS | OUTER 0 SEQ TBD 1 created: Train - 152 rows, 152 groups | Dev - 76 rows, 76 groups
INFO | CREATING_DATASPLITS | OUTER 0 SEQ TBD 2 created: Train - 152 rows, 152 groups | Dev - 76 rows, 76 groups
INFO | PREPARE_EXECUTION | Calculating MRMR feature sets...
INFO | PREPARE_EXECUTION | Running Optuna Optimization with a global HP set
INFO | PREPARE_EXECUTION | A new study created in memory with name: optuna_0_0


...

INFO | OPTUNA | Trial 3 finished with value: 0.9549295774647888 and parameters: {'n_outliers': 0, 'n_mrmr_features': 30, 'max_depth_ExtraTreesClassifier': 21, 'min_samples_split_ExtraTreesClassifier': 16, 'min_samples_leaf_ExtraTreesClassifier': 48, 'max_features_ExtraTreesClassifier': 0.5696634895750645, 'n_estimators_ExtraTreesClassifier': 266, 'class_weight_ExtraTreesClassifier': 'balanced'}
INFO | OPTUNA | Inner sequential training completed for bag_id 1_0_4 and training id 1_0_0
INFO | OPTUNA | Inner sequential training completed for bag_id 1_0_4 and training id 1_0_1
INFO | OPTUNA | Inner sequential training completed for bag_id 1_0_4 and training id 1_0_2
INFO | OPTUNA | Bag 1_0_4 training summary: 3/3 successful, 0 failed
INFO | SCORES | Trial scores for metric: AUCROC
INFO | SCORES | Scores: train_avg: 0.989 | dev_avg: 0.987 | test_avg: 0.978 | train_ensemble: 0.990 | dev_ensemble: 0.981 | test_ensemble: 0.979
INFO | SCORES | train_lst: [0.990925925925926, 0.9869861598843214, 

[I 2026-04-10 16:47:40,625] Trial 4 finished with value: 0.9806959403479703 and parameters: {'n_outliers': 1, 'n_mrmr_features': 30, 'max_depth_ExtraTreesClassifier': 19, 'min_samples_split_ExtraTreesClassifier': 3, 'min_samples_leaf_ExtraTreesClassifier': 31, 'max_features_ExtraTreesClassifier': 0.6508861504501793, 'n_estimators_ExtraTreesClassifier': 347, 'class_weight_ExtraTreesClassifier': None}. Best is trial 1 with value: 0.9885666942833472.


INFO | OPTUNA | Trial 4 finished with value: 0.9806959403479703 and parameters: {'n_outliers': 1, 'n_mrmr_features': 30, 'max_depth_ExtraTreesClassifier': 19, 'min_samples_split_ExtraTreesClassifier': 3, 'min_samples_leaf_ExtraTreesClassifier': 31, 'max_features_ExtraTreesClassifier': 0.6508861504501793, 'n_estimators_ExtraTreesClassifier': 347, 'class_weight_ExtraTreesClassifier': None}
INFO | SCORES | Optimization results: 
INFO | SCORES | Best trial number 1
INFO | SCORES | Best target value 0.9885666942833472
INFO | SCORES | Best parameters {'outl_reduction': 3, 'n_input_features': 30, 'ml_model_type': <ModelName.ExtraTreesClassifier: 'ExtraTreesClassifier'>, 'ml_model_params': {'max_depth': 13, 'min_samples_split': 80, 'min_samples_leaf': 27, 'max_features': 0.6112401049845391, 'n_estimators': 471, 'class_weight': 'balanced', 'criterion': 'entropy', 'bootstrap': True, 'n_jobs': 1, 'random_state': 0}, 'positive_class': 1}
INFO | SCORES | Performance Info: {'train_avg': 0.9896848892

/Users/M228393/github/Octopus_public/octopus/utils.py:93: UserWarning: The DataFrame has column names of mixed type. They will be converted to strings and not roundtrip correctly.
  table = pa.Table.from_pandas(df, preserve_index=index)


INFO | RESULTS | Study completed. Results saved to: studies/predict_example-20260410_144723
Study saved to: studies/predict_example-20260410_144723


## 2. Load the Study


In [6]:
from octopus.predict.study_io import load_study_info

study_info = load_study_info(study.output_path)

print(f"ML type: {study_info.config['ml_type']}")
print(f"Target metric: {study_info.config['target_metric']}")
print(f"Outer splits: {len(study_info.outer_split_dirs)}")
print(f"Features: {len(study_info.config['feature_cols'])}")

ML type: binary
Target metric: AUCROC
Outer splits: 2
Features: 30


## 3. Create an OctoPredictor


In [7]:
from octopus.predict import OctoPredictor

tp = OctoPredictor(study=study_info, task_id=0)
print(f"Loaded models for {len(tp._models)} outer splits")

Loaded models for 2 outer splits


## 4. Predict on New Data

Each outer-split model predicts independently, then results are averaged into an ensemble prediction. Use `per_split=True` to see individual split predictions.


In [9]:
# Ensemble-averaged predictions (row_id + prediction)
predictions = tp.predict(new_df, per_split=True)
predictions.head(10)

,row_id,prediction,split_0,split_1
0,0,0.0,0.0,0.0
1,1,1.0,1.0,1.0
2,2,0.0,0.0,0.0
3,3,0.5,1.0,0.0
4,4,0.0,0.0,0.0
5,5,1.0,1.0,1.0
6,6,1.0,1.0,1.0
7,7,0.0,0.0,0.0
8,8,0.0,0.0,0.0
9,9,0.0,0.0,0.0


In [10]:
# With per-split predictions as additional columns
predictions_detail = tp.predict(new_df, per_split=True)
predictions_detail.head(10)

,row_id,prediction,split_0,split_1
0,0,0.0,0.0,0.0
1,1,1.0,1.0,1.0
2,2,0.0,0.0,0.0
3,3,0.5,1.0,0.0
4,4,0.0,0.0,0.0
5,5,1.0,1.0,1.0
6,6,1.0,1.0,1.0
7,7,0.0,0.0,0.0
8,8,0.0,0.0,0.0
9,9,0.0,0.0,0.0


## 5. Predict Probabilities

For classification tasks, `predict_proba` returns class probabilities.


In [11]:
# Ensemble-averaged probabilities (row_id + one column per class)
proba = tp.predict_proba(new_df)
proba.head(10)

,row_id,0,1
0,0,0.769965,0.230035
1,1,0.111642,0.888358
2,2,0.661370,0.338630
3,3,0.486047,0.513953
4,4,0.758067,0.241933
5,5,0.237224,0.762776
6,6,0.165245,0.834755
7,7,0.767734,0.232266
8,8,0.754870,0.245130
9,9,0.776588,0.223412


In [12]:
# With per-split probabilities as additional columns
proba_detail = tp.predict_proba(new_df, per_split=True)
proba_detail.head(10)

,row_id,0,1,0_split_0,1_split_0,0_split_1,1_split_1
0,0,0.769965,0.230035,0.706793,0.293207,0.833136,0.166864
1,1,0.111642,0.888358,0.095758,0.904242,0.127526,0.872474
2,2,0.661370,0.338630,0.603109,0.396891,0.719632,0.280368
3,3,0.486047,0.513953,0.411730,0.588270,0.560363,0.439637
4,4,0.758067,0.241933,0.690868,0.309132,0.825266,0.174734
5,5,0.237224,0.762776,0.177542,0.822458,0.296907,0.703093
6,6,0.165245,0.834755,0.161854,0.838146,0.168637,0.831363
7,7,0.767734,0.232266,0.703820,0.296180,0.831648,0.168352
8,8,0.754870,0.245130,0.688977,0.311023,0.820762,0.179238
9,9,0.776588,0.223412,0.714466,0.285534,0.838710,0.161290


## 6. Evaluate Performance

Score each outer-split model on the held-out new data.


In [13]:
perf = tp.performance(new_df, metrics=["AUCROC", "ACCBAL"])
perf

,outer_split,metric,score
0,0,AUCROC,0.989749
1,0,ACCBAL,0.919643
2,1,AUCROC,0.987765
3,1,ACCBAL,0.913690


## 7. Save and Load for Deployment

Save the predictor to a standalone directory (models + metadata only, no data). Load it back and verify predictions match.


In [14]:
# Save to a temporary directory
save_dir = tempfile.mkdtemp(prefix="octopus_predictor_")
tp.save(save_dir)
print(f"Saved predictor to: {save_dir}")
print(f"Contents: {os.listdir(save_dir)}")

Saved predictor to: /var/folders/lr/x666zkrd2mn_6l3v8kg0kv_h0000gn/T/octopus_predictor_u2k5mpzn
Contents: ['selected_features', 'version.json', 'models', 'study_config.json', 'predictor_state.json']


In [15]:
# Load the saved predictor (no study directory needed)
tp_loaded = OctoPredictor.load(save_dir)

# Verify predictions match
predictions_loaded = tp_loaded.predict(new_df)
pd.testing.assert_frame_equal(predictions, predictions_loaded)
print("Predictions match after save/load.")

NameError: name 'pd' is not defined